# HP-3 · Huấn luyện mô hình dự đoán thời lượng

**Mục tiêu.** Từ CSV của notebook `01`, học một mô hình đoán *câu này đọc mất mấy
giây* — để vòng lặp ngân sách (HP-5) biết trước câu nào vượt khung mà rút gọn
**trước**, thay vì nén âm thanh cho gấp gáp **sau**.

**Mô hình.** Hồi quy tuyến tính có ridge trên vài đặc trưng đếm được của câu
(số âm tiết, số ký tự, số từ, dấu câu…). Cố tình chọn mô hình nhỏ: chạy tức thì,
lưu ra một file JSON vài KB, không cần GPU lúc dùng, và giải thích được từng hệ số.

**So sánh với.** Ước lượng thô `số âm tiết / 4,3 + 0,3s` — mốc phải vượt qua.

**Mục tiêu đề cương (MT-3).** MAE < 200 ms.

In [ ]:
# Chạy được dù mở notebook từ thư mục nào: đi ngược lên tới thư mục có backend/
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "backend").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Gốc dự án:", ROOT)

In [ ]:
import csv
import random

from backend.core import duration

DATA = ROOT / "data" / "duration" / "samples.csv"
TEST_RATIO = 0.1
RIDGE = 1e-2
SEED = 13

with DATA.open(encoding="utf-8", newline="") as fh:
    rows = [(r["text"], float(r["seconds"])) for r in csv.DictReader(fh) if r.get("text")]

print(f"{len(rows)} mẫu trong {DATA}")
print("Đặc trưng dùng:", ", ".join(duration.FEATURE_NAMES))

## Chia tập huấn luyện / kiểm tra\n\nXáo với seed cố định để chạy lại ra đúng kết quả cũ.

In [ ]:
assert len(rows) >= 10, f"Chỉ có {len(rows)} mẫu — cần ít nhất 10 để chia tập kiểm tra."

data = list(rows)
random.Random(SEED).shuffle(data)
n_test = max(1, int(len(data) * TEST_RATIO))
test, train = data[:n_test], data[n_test:]
print(f"Huấn luyện {len(train)} câu · kiểm tra {len(test)} câu")

## Huấn luyện

In [ ]:
model = duration.DurationModel.fit(
    [t for t, _ in train], [s for _, s in train], ridge=RIDGE
)

print("Hệ số học được:")
for name, w in zip(duration.FEATURE_NAMES, model.weights):
    print(f"  {name:>12}: {w:+.4f}")

## Đánh giá trên tập kiểm tra

In [ ]:
rep = duration.evaluate(model, [t for t, _ in test], [s for _, s in test])

print(f"MAE mô hình      : {rep['mae_ms']:.0f} ms   (RMSE {rep['rmse_ms']:.0f} ms)")
print(f"MAE ước lượng thô: {rep['heuristic_mae_ms']:.0f} ms")
print(f"Cải thiện        : {rep['improvement'] * 100:.1f}%")
print()

if len(test) < 100:
    print(f"⚠ Tập kiểm tra chỉ {len(test)} câu — con số này CHƯA nói lên gì về MT-3.")
    print("  Đề cương cần 30.000–50.000 câu rồi mới kết luận được.")
else:
    print("Mục tiêu MT-3 (MAE < 200 ms):", "ĐẠT" if rep["mae_ms"] < 200 else "chưa đạt")

## Xem sai số ở đâu

In [ ]:
texts = [t for t, _ in test]
truth = [s for _, s in test]
pred = model.predict_many(texts)

worst = sorted(zip(texts, truth, pred), key=lambda x: -abs(x[1] - x[2]))[:10]
print("10 câu lệch nhiều nhất:\n")
print(f'{"thật":>7} {"đoán":>7} {"lệch":>7}  câu')
for t, y, p in worst:
    print(f"{y:7.2f} {p:7.2f} {p - y:+7.2f}  {t[:50]}")

In [ ]:
# Biểu đồ đoán vs thật — có matplotlib thì vẽ, không có thì bỏ qua.
try:
    import matplotlib.pyplot as plt

    lo, hi = min(truth + pred), max(truth + pred)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(truth, pred, s=18, alpha=0.7)
    ax.plot([lo, hi], [lo, hi], "--", linewidth=1)
    ax.set_xlabel("Thời lượng đo thật (giây)")
    ax.set_ylabel("Thời lượng mô hình đoán (giây)")
    ax.set_title(f"HP-3 · MAE {rep['mae_ms']:.0f} ms trên {len(test)} câu")
    fig.tight_layout()
    plt.show()
except ImportError:
    print("Chưa có matplotlib (không bắt buộc). Cài: uv pip install matplotlib")

## Lưu mô hình\n\nLưu vào `data/models/duration_vi.json` — chỗ mà `duration.default_model()` tự nạp khi chạy lồng tiếng.

In [ ]:
model.meta.update({"test": rep, "data": str(DATA), "n_train": len(train), "n_test": len(test)})
path = model.save(duration.MODEL_PATH)
print("Đã lưu ->", path)

# Nạp lại để chắc chắn dùng được
duration.reset_default()
m = duration.default_model()
print("Mô hình mặc định đã huấn luyện:", m.trained)
print("Thử:", f'{m.predict("Xin chào các bạn, hôm nay trời rất đẹp."):.2f}s')